# Задача 4. Пайплайн прогнозирования

Offline-пайплайн `SolarForecastPipeline` (`src/pipeline.py`).

**Модель по умолчанию:** `RandomForestRegressor` — лучший RMSE на hold-out (~4356) при использовании погодных exog; fit ~0.4 с.

**Альтернативы:** `SeasonalNaive` — лучший sMAPE (19%) на полном ряду; `AutoTheta` — лучший RMSE на rolling backtest (stats); `AutoETS` — лучший RMSE среди stats на hold-out.

**Этапы:**
1. Подготовка данных (`build_datasets`) → `__output__/train.parquet`, `test.parquet`
2. Обучение выбранной модели на train (stats / ML / DL)
3. Прогноз на 48 ч для серий Plant 1 и Plant 2
4. Метрики на hold-out (all + `no_zeros`), сохранение артефактов

**Вывод по задаче 4** — см. раздел 6 в `README.md`.


In [1]:
import json

import pandas as pd

from src.config import DEFAULT_MODEL, OUTPUT_DIR
from src.data import build_datasets
from src.metrics import evaluate
from src.pipeline import SolarForecastPipeline


In [2]:
train_df, test_df = build_datasets(save=True)
print(f'Train: {len(train_df)} rows, Test: {len(test_df)} rows')
print(f'Series: {train_df["unique_id"].unique().tolist()}')

pipe = SolarForecastPipeline(model_name=DEFAULT_MODEL)
result = pipe.run(train_df=train_df, test_df=test_df)

print('Model:', result.model_name)
print('Metrics (all):', result.metrics)

eval_df = pd.read_parquet(OUTPUT_DIR / 'pipeline' / 'latest_evaluation.parquet')
model_col = result.model_name
print('Metrics (no zeros):', result.metrics_no_zeros)
print(f'Fit: {result.fit_seconds:.2f}s, Predict: {result.predict_seconds:.3f}s')
result.metrics


Train: 1536 rows, Test: 96 rows
Series: ['1', '2']
Model: RandomForestRegressor
Metrics (all): {'MAE': 1805.60265562758, 'RMSE': 4356.047973853025, 'MAPE': 6.759903172210606, 'sMAPE': 5.682653162021147}
Metrics (no zeros): {'MAE': 3333.420287312455, 'RMSE': 5918.703696511054, 'MAPE': 12.479821241004194, 'sMAPE': 10.491051991423655}
Fit: 0.32s, Predict: 1.979s


{'MAE': 1805.60265562758,
 'RMSE': 4356.047973853025,
 'MAPE': 6.759903172210606,
 'sMAPE': 5.682653162021147}

In [3]:
configs = [
    'Naive',
    'SeasonalNaive',
    'AutoTheta',
    'AutoETS',
    'AutoARIMA',
    'RandomForestRegressor',
    'NHITS',
]
bench = []
bench_nz = []
for name in configs:
    r = SolarForecastPipeline(model_name=name).run(train_df=train_df, test_df=test_df)
    eval_df = pd.read_parquet(OUTPUT_DIR / 'pipeline' / 'latest_evaluation.parquet')
    metrics_nz = evaluate(eval_df['y'].values, eval_df[name].values, no_zeros=True)
    bench.append({
        'model': name,
        **r.metrics,
        'fit_s': r.fit_seconds,
        'predict_s': r.predict_seconds,
    })
    bench_nz.append({
        'model': name,
        **metrics_nz,
        'fit_s': r.fit_seconds,
        'predict_s': r.predict_seconds,
    })

bench_df = pd.DataFrame(bench).set_index('model').sort_values('RMSE')
bench_nz_df = pd.DataFrame(bench_nz).set_index('model').sort_values('RMSE')

print('Hold-out (all observations)')
display(bench_df)
print('Hold-out (y != 0)')
display(bench_nz_df)


Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 2.6 M  | train
-------------------------------------------------------
2.6 M     Trainable params
0         Non-trainable params
2.6 M     Total params
10.372    Total estimated model params size (

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\GitProjects\TSA\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

Hold-out (all observations)


,MAE,RMSE,MAPE,sMAPE,fit_s,predict_s
model,,,,,,
RandomForestRegressor,1805.602656,4356.047974,6.759903e+00,5.682653,0.318925,1.700746
AutoETS,5267.594130,8682.154858,6.559906e+10,109.440486,2.096027,1.961930
AutoTheta,4910.524014,9048.034779,3.615114e+09,106.683412,2.117629,2.008658
NHITS,5058.468083,9128.738921,6.618635e+09,108.497918,5.905270,0.087403
AutoARIMA,5266.542884,9977.902462,3.505419e+09,108.172259,185.031887,1.925134
SeasonalNaive,5860.657587,10989.593847,2.864880e+01,18.980735,1.969268,1.949395
Naive,19887.051661,32577.206719,5.416667e+01,108.333333,2.040330,2.058540


Hold-out (y != 0)


,MAE,RMSE,MAPE,sMAPE,fit_s,predict_s
model,,,,,,
RandomForestRegressor,3333.420287,5918.703697,12.479821,10.491052,0.318925,1.700746
AutoETS,8513.729618,11703.569734,40.196744,32.813206,2.096027,1.961930
AutoTheta,8998.842234,12293.510957,41.689115,27.723221,2.117629,2.008658
NHITS,9216.520116,12402.288898,43.756097,31.073080,5.905270,0.087403
AutoARIMA,9658.132979,13556.864014,46.383372,30.471862,185.031887,1.925134
SeasonalNaive,10819.675546,14931.917674,52.890096,35.041357,1.969268,1.949395
Naive,36714.556913,44263.707605,100.000000,200.000000,2.040330,2.058540


In [4]:
artifact_dir = OUTPUT_DIR / 'pipeline'
print('Artifacts:')
for p in sorted(artifact_dir.glob('*')):
    print('-', p.name)
    if p.suffix == '.json':
        print(json.loads(p.read_text(encoding='utf-8')))

Artifacts:
- latest_evaluation.parquet
- latest_forecast.parquet
- metrics.json
{'model': 'NHITS', 'horizon': 48, 'metrics': {'MAE': 5058.46808303249, 'RMSE': 9128.73892126122, 'MAPE': 6618635368.206529, 'sMAPE': 108.49791825685851}, 'metrics_no_zeros': {'MAE': 9216.52011616142, 'RMSE': 12402.288898071361, 'MAPE': 43.75609668146081, 'sMAPE': 31.073079858815717}, 'fit_seconds': 5.905269900000349, 'predict_seconds': 0.08740299999954004}


### Обоснование настройки пайплайна

| Компонент | Выбор | Причина |
|-----------|-------|---------|
| Модель по умолчанию | `RandomForestRegressor` | RMSE ~4356 на hold-out; fit ~0.3 с |
| Baseline sMAPE | `SeasonalNaive` | sMAPE 19% на полном ряду |
| Stats (backtest) | `AutoTheta` | Лучший RMSE rolling CV |
| Stats (hold-out) | `AutoETS` | RMSE 8682 |
| DL (опционально) | `NHITS` | RMSE 8473, fit ~11 с |
| Горизонт | 48 ч | Постановка задачи |
| Метрики | all + `no_zeros` | Ночные нули искажают MAPE |

**Вывод по задаче 4** и **общее заключение** — `README.md`, разделы 6 и 8.
